# AION: RAG Pipeline Proof of Concept

This notebook demonstrates a complete, working RAG (Retrieval-Augmented Generation) pipeline for the AION project. It covers two main stages:

1.  **Ingestion**: Loading documents from the `data/` directory, splitting them into chunks, generating embeddings with Google's Vertex AI, and storing them in a local ChromaDB vector store.
2.  **Retrieval**: Taking a user query, generating an embedding for it, and retrieving the most semantically relevant document chunks from the vector store.

### Step 1: Setup and Dependencies

First, we import all necessary libraries. Note that authentication with Google Cloud (for Vertex AI) must be configured in your environment for this to work.

In [2]:
import chromadb
from pathlib import Path

# LangChain components
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_vertexai import VertexAIEmbeddings

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


## Stage 1: Data Ingestion

Here, we initialize our `FileLoader` and `ChromaManager`. We then load the sample log and OCPP spec files, split them into manageable chunks, and add them to the vector store. The `ChromaManager` will handle the process of calling the Vertex AI API to generate embeddings for each chunk.

In [7]:
class FileLoader:
    """
    Loads and processes text files from a directory.
    """
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )

    def load_and_split_documents(self, data_path: str | Path) -> list:
        path = Path(data_path)
        if not path.is_dir():
            raise ValueError(f"Provided data path '{data_path}' is not a directory.")

        all_docs = []
        for file_path in path.glob("**/*"):
            if file_path.is_file() and file_path.suffix in [".txt", ".log", ".md"] and file_path.name not in ["README.md", "dataReadMe.md"]:
                print(f"📄 Loading file: {file_path.name}")
                loader = TextLoader(str(file_path), encoding="utf-8")
                docs = loader.load()
                all_docs.extend(docs)

        print(f"\nSplitting {len(all_docs)} documents into chunks...")
        return self.text_splitter.split_documents(all_docs)

class ChromaManager:
    """
    A manager class for interacting with a local ChromaDB vector store using Vertex AI.
    """
    def __init__(self, path: str = "./chroma_db_notebook", collection_name: str = "aion_docs"):
        self.embedding_model = VertexAIEmbeddings(model_name="text-embedding-004", project="analog-signal-437916-c0", location="us-central1")
        self.client = chromadb.PersistentClient(path=path)
        self.collection = self.client.get_or_create_collection(name=collection_name)
        print(f"✅ ChromaDB Manager initialized. Collection '{collection_name}' ready.")

    def add_documents(self, documents: list, metadatas: list, ids: list):
        if not documents:
            print("No documents to add.")
            return

        print(f"📦 Adding {len(documents)} document chunks to the vector store...")
        print("⚡ Generating embeddings with Vertex AI...")
        embeddings = self.embedding_model.embed_documents(documents)

        self.collection.add(
            embeddings=embeddings, documents=documents, metadatas=metadatas, ids=ids
        )
        print("✅ Documents added successfully.")

    def query(self, query_text: str, n_results: int = 3) -> list:
        print(f"⚡ Generating query embedding with Vertex AI for: '{query_text}'")
        query_embedding = self.embedding_model.embed_query(query_text)

        results = self.collection.query(
            query_embeddings=[query_embedding], n_results=n_results
        )
        return results["documents"][0] if results and results["documents"] else []


# --- INGESTION PIPELINE ---

# 1. Initialize components
file_loader = FileLoader()
chroma_manager = ChromaManager()

# 2. Load and process documents from the 'data/' directory (relative to notebook)
documents = file_loader.load_and_split_documents(data_path="../data/")

# 3. Prepare data for ChromaDB
contents = [doc.page_content for doc in documents]
metadatas = [doc.metadata for doc in documents]
ids = [f"{doc.metadata['source']}-{i}" for i, doc in enumerate(documents)]

# 4. Add to vector store (this step calls the Vertex AI API)
chroma_manager.add_documents(documents=contents, metadatas=metadatas, ids=ids)

✅ ChromaDB Manager initialized. Collection 'aion_docs' ready.
📄 Loading file: successful_session.log
📄 Loading file: charging_session_error.log
📄 Loading file: ocpp_2.0.1_charging_management.md

Splitting 3 documents into chunks...
📦 Adding 8 document chunks to the vector store...
⚡ Generating embeddings with Vertex AI...


/Users/chayan/Developer/chargepoint-emu/aion-poc/.venv/lib/python3.11/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ Documents added successfully.


## Stage 2: Retrieval

Now that our knowledge base is built, we can test the retrieval process. We'll define a query related to the content of our sample files. The `ChromaManager` will use the same Vertex AI model to embed our query and find the most similar documents in the database.

In [8]:
query = "Why did the charging station fail to authenticate?"

print(f"❓ Querying the vector store with: '{query}'\n")

# The query method handles embedding the query text and finding similar documents
retrieved_docs = chroma_manager.query(query, n_results=2)

if retrieved_docs:
    print("✨ Top retrieved documents: ✨")
    for i, doc in enumerate(retrieved_docs):
        print(f"\n--- Document {i + 1} ---")
        print(doc)
else:
    print("❌ No relevant documents were found.")

❓ Querying the vector store with: 'Why did the charging station fail to authenticate?'

⚡ Generating query embedding with Vertex AI for: 'Why did the charging station fail to authenticate?'
✨ Top retrieved documents: ✨

--- Document 1 ---
2024-11-02 10:15:23 [ERROR] Charging session failed for connector 1
2024-11-02 10:15:23 [INFO] Station ID: CP001, Connector: 1, RFID: 1234567890
2024-11-02 10:15:24 [ERROR] Ground fault detected on connector 1
2024-11-02 10:15:24 [ERROR] Emergency stop triggered due to ground fault
2024-11-02 10:15:25 [INFO] Charging session terminated
2024-11-02 10:15:25 [ERROR] Connector 1 status changed to FAULTED
2024-11-02 10:15:26 [INFO] Notification sent to management system

--- Document 2 ---
2024-11-02 11:30:15 [INFO] Charging session started for connector 2
2024-11-02 11:30:15 [INFO] Station ID: CP001, Connector: 2, RFID: 9876543210
2024-11-02 11:30:16 [INFO] Vehicle connected, authentication successful
2024-11-02 11:30:17 [INFO] Charging started at 7.4 kW


In [9]:
query_2 = "What is the purpose of a heartbeat message?"

print(f"❓ Querying the vector store with: '{query_2}'\n")

retrieved_docs_2 = chroma_manager.query(query_2, n_results=2)

if retrieved_docs_2:
    print("✨ Top retrieved documents: ✨")
    for i, doc in enumerate(retrieved_docs_2):
        print(f"\n--- Document {i + 1} ---")
        print(doc)

❓ Querying the vector store with: 'What is the purpose of a heartbeat message?'

⚡ Generating query embedding with Vertex AI for: 'What is the purpose of a heartbeat message?'
✨ Top retrieved documents: ✨

--- Document 1 ---
2024-11-02 12:30:47 [INFO] Session duration: 2 hours 15 minutes
2024-11-02 12:30:48 [INFO] Connector 2 status changed to AVAILABLE

--- Document 2 ---
2024-11-02 10:15:27 [DEBUG] Error code: GF_001 - Ground fault protection activated
2024-11-02 10:15:28 [INFO] Maintenance required for connector 1
